# OmniParser smoke test (separate runtime from JEDI)

Runs the three OmniParser variants (`omniparser.py`, `omniparser2.py`, `omniparser_structural.py`) on 5 DesignBench samples via `scripts/prod_omniparser.py`.

**Separate runtime from JEDI.** Florence-2 (OmniParser's caption model) needs `transformers==4.49.0`, which conflicts with `vllm>=0.8.3` (required for JEDI). We split: this notebook runs OmniParser only.

Output JSON + annotated PNGs land in `/content/drive/MyDrive/omniparser-test/`.

**Prereqs:** Colab Pro A100, HF read token (only if weights aren't cached), OmniParser weights already downloaded to `/content/drive/MyDrive/omniparser-weights/` (from jedi_smoke cell 3 — or this notebook downloads them if missing).

## Cell 1 — Bootstrap: Drive mount, repo clone, pinned deps

Installs `transformers==4.49.0` (Florence-2 compat) + `Pillow<11` (`_Ink` still exists) + `easyocr`/`safetensors`/`ultralytics`. No vllm here.

**After first run, RESTART RUNTIME** (once, because of `--force-reinstall`), then re-run from cell 1.

In [1]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/isaacau502/GUI-grounded-gen'
REPO_DIR = '/content/GUI-grounded-gen'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    head = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    print(f'Cloned fresh @ {head}')
else:
    before = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    after = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    if before == after:
        print(f'No new commits (HEAD: {after})')
    else:
        print(f'Pulled {before} -> {after}:')
        log = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '--oneline', f'{before}..{after}']).decode().strip()
        print(log)

# Pinned for Florence-2 / OmniParser. NO vllm in this notebook.
# transformers==4.49.0 - Florence-2 trust_remote_code needs this exact range
# Pillow<11         - transformers 4.49 imports _Ink from PIL._typing (removed in Pillow 11)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.49.0',
                'Pillow<11',
                'qwen-vl-utils',
                'huggingface_hub',
                'hf_transfer',
                'safetensors',
                'ultralytics',
                'easyocr',
                'matplotlib',
                '--force-reinstall'], check=True)

print('Bootstrap complete.')
print('If this is a fresh runtime: RESTART RUNTIME now, then re-run cells 1+.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
No new commits (HEAD: 0c226f7)
Bootstrap complete.
If this is a fresh runtime: RESTART RUNTIME now, then re-run cells 1+.


## Cell 2 — HF token (only if weights not cached)

Most OmniParser models are public, but `getpass` here in case Drive cache is empty. Skip if `/content/drive/MyDrive/omniparser-weights/` already has `icon_detect/` + `icon_caption_florence/`.

In [2]:
import os, getpass

if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = getpass.getpass('HF token (blank to skip): ')
    except Exception:
        pass

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import HfApi
    me = HfApi().whoami()
    print(f"HF authed as: {me.get('name', '?')}")
else:
    print('No HF token; relying on anonymous access + Drive cache.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


HF authed as: isaacau502


## Cell 3 — Ensure OmniParser weights are on Drive

Idempotent. If already at `/content/drive/MyDrive/omniparser-weights/`, skips. If missing, downloads (~1.3GB) via `hf_transfer`. Matches jedi_smoke cell 3's output layout so both notebooks see the same cache.

In [3]:
import os, shutil
from pathlib import Path

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OMNI_DRIVE = '/content/drive/MyDrive/omniparser-weights'
OMNI_LOCAL = '/content/omniparser-weights'

def fetch(repo_id, local_dir, drive_dir, sentinel='config.json'):
    # Check both possible nested paths
    if any(os.path.exists(os.path.join(drive_dir, d, sentinel))
           for d in ['icon_caption_florence', 'icon_caption']):
        print(f'{repo_id}: already at {drive_dir}, skipping.')
        return
    from huggingface_hub import snapshot_download
    print(f'{repo_id}: downloading to {local_dir}...')
    snapshot_download(repo_id=repo_id, local_dir=local_dir,
                      resume_download=True,
                      token=os.environ.get('HF_TOKEN'))
    print(f'{repo_id}: copying to {drive_dir}...')
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    print(f'{repo_id}: done.')

fetch('microsoft/OmniParser-v2.0', OMNI_LOCAL, OMNI_DRIVE)

# Rename icon_caption -> icon_caption_florence if needed (wrapper expects this name)
src = Path(OMNI_DRIVE) / 'icon_caption'
dst = Path(OMNI_DRIVE) / 'icon_caption_florence'
if src.exists() and not dst.exists():
    src.rename(dst)
    print(f'Renamed {src.name} -> {dst.name}')

print('\nContents of', OMNI_DRIVE)
for p in sorted(Path(OMNI_DRIVE).iterdir()):
    print(f'  {p.name}/')

microsoft/OmniParser-v2.0: already at /content/drive/MyDrive/omniparser-weights, skipping.

Contents of /content/drive/MyDrive/omniparser-weights
  .cache/
  .gitattributes/
  README.md/
  config.json/
  handler.py/
  icon_caption_florence/
  icon_detect/
  requirements.txt/


## Cell 4 — Run prod_omniparser

Iterates all three variants (v1 / v2 / structural) over 5 DesignBench samples (one per defect type). Saves JSON results + annotated PNGs to `/content/drive/MyDrive/omniparser-test/`.

~5-8 min on A100. First call downloads Florence-2 base (~460MB) if not cached.

In [4]:
!cd /content/GUI-grounded-gen && git pull
!python /content/GUI-grounded-gen/scripts/prod_omniparser.py

Already up to date.
Loading v1 (OmniParser)...
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
E0000 00:00:1776673474.371524    4502 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776673474.378855    4502 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776673474.396411    4502 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776673474.396443    4502 computation_placer.cc:177] computation placer already registered. Please che

## Cell 5 — What worked / how to use

### Canonical code
- [`grounding/omniparser.py`](../grounding/omniparser.py) — v1, keyword match returns single click point
- [`grounding/omniparser2.py`](../grounding/omniparser2.py) — v2, full element list with bboxes + captions, `prompt_block` ready for Qwen72B
- [`grounding/omniparser_structural.py`](../grounding/omniparser_structural.py) — YOLO + OCR + geometric relations, no defect-specific logic
- [`scripts/prod_omniparser.py`](../scripts/prod_omniparser.py) — this notebook's workhorse

### Why a separate notebook from JEDI?

Florence-2 (inside OmniParser) was written against `transformers==4.49.0` and Microsoft hasn't updated it. Any transformers 4.50+ breaks Florence-2's `trust_remote_code` loading.

vllm (used by JEDI) requires `transformers>=4.56` since vllm 0.8.3. vllm 0.8.2 (which accepts transformers 4.49) has a JEDI/Qwen2.5-VL worker crash bug.

**Net result:** can't fit both in one runtime. Split notebooks, run serially, pipe through Drive JSON cache.

### Orchestration order
1. **`colab/jedi_smoke.ipynb`** → produces JEDI click coords per sample → saves to Drive JSON
2. **`colab/omniparser_smoke.ipynb`** (this one) → produces OmniParser element lists → saves to Drive JSON
3. **Future: eval notebook** → reads both JSON caches → calls Qwen72B with grounding injected → computes DesignBench metrics